# 🧩 Building Useful AI Agents with Agno + OpenRouter
## Notebook 1 — Foundations (Basic Session)

Welcome! This is the **hands-on companion** to the workshop slides. We'll run real code
alongside the presentation, building up an AI agent piece by piece.

This first notebook covers the **foundations** (slides 1–10):

| # | Topic | What you'll learn |
|---|-------|-------------------|
| 1 | **What is Agno?** | The mental model: Model + Instructions + Tools = Agent |
| 2 | **Your First Agent** | The core request → response loop |
| 3 | **Tools** | Giving agents the ability to *act* (custom + built-in) |
| 4 | **Structured Outputs** | Reliable, machine-readable responses with Pydantic |
| 5 | **Teams** | Specialised agents collaborating on a task |

> 🧵 **Running example:** we'll build **"Safari"**, a travel-planning assistant, and grow
> it across every section so the concepts connect into one coherent story.

The **advanced session** (Memory, Knowledge, Storage, Workflows) continues in
`02_agno_advanced.ipynb`.


---
## 0. Setup

This project is managed with [**uv**](https://docs.astral.sh/uv/) — a fast Python package
manager. The environment is already defined in `pyproject.toml`. If you're starting fresh,
this is all it takes:

```bash
# create the project + virtual env and install everything
uv sync

# register the venv as a Jupyter kernel (once)
uv run python -m ipykernel install --user --name agno-workshop
```

Then select the **agno-workshop** kernel in Jupyter (top-right of the notebook).

The key packages: `agno` (the agent framework), `openai` (used under the hood by the
OpenRouter model class), `lancedb` + `fastembed` (for the RAG section in notebook 2),
and `sqlalchemy` (for storage).


### 🔑 OpenRouter API key

We use [**OpenRouter**](https://openrouter.ai) as our model gateway. One API key gives us
access to **hundreds of models** (OpenAI, Google, Meta, Anthropic, …) behind a single
OpenAI-compatible API — perfect for a workshop where everyone can pick a cheap model.

1. Sign up at https://openrouter.ai and create a key under **Keys**.
2. Create a file called `.env` next to this notebook (copy `.env.example`) and add:

   ```
   OPENROUTER_API_KEY=sk-or-v1-...........
   ```

The cell below loads that file. If no key is found, it will prompt you to paste one
securely (it won't be echoed or saved).


In [ ]:
from dotenv import load_dotenv
import os, getpass

load_dotenv()  # reads a local .env file if present

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

print("✅ OpenRouter key loaded" if os.getenv("OPENROUTER_API_KEY") else "❌ No key found")


### 🧠 Choosing a model

OpenRouter model IDs look like `provider/model`. We keep the choice in **one variable**
so you can swap models everywhere at once.

A few **cheap, capable** options (great for a workshop):

| Model ID | Notes |
|----------|-------|
| `openai/gpt-4o-mini` | Cheap, fast, reliable **tool-calling + structured output** (default) |
| `google/gemini-2.0-flash-001` | Very cheap, fast, large context |
| `meta-llama/llama-3.3-70b-instruct` | Strong open model, low cost |

> 💡 For the **Tools** and **Structured Outputs** sections, pick a model that supports
> function calling and JSON output. `gpt-4o-mini` is the safest default. Browse prices at
> https://openrouter.ai/models.


In [ ]:
from agno.models.openrouter import OpenRouter

# 👇 Swap this one line to use a different model anywhere in the notebook.
MODEL_ID = "openai/gpt-4o-mini"

def make_model():
    """A fresh model client pointed at OpenRouter."""
    return OpenRouter(id=MODEL_ID)

print(f"Using model: {MODEL_ID}")


---
## 1. What is Agno?  *(slides 1–3)*

**Agno** is a lightweight, **model-agnostic** framework for building AI agents. Its
philosophy is *batteries-included*: Agents, Tools, Memory, Knowledge, Storage and
Workflows all ship in the box.

An **Agent** is the core unit, and it's just three things combined:

```
            ┌─────────────┐
            │    Model    │   the reasoning engine (any LLM via OpenRouter)
            └──────┬──────┘
                   │
   Instructions ───┼─── Tools
   (who it is &    │    (what it can DO:
    how to behave) │     APIs, code, search…)
                   ▼
              Agno Agent
```

| Pillar | Meaning |
|--------|---------|
| **Model agnostic** | Swap the LLM without changing your agent logic |
| **Fast runtime** | Minimal overhead — agents instantiate in microseconds |
| **Batteries included** | Tools, memory, knowledge & storage are built in |

Enough theory — let's build one.


---
## 2. Your First Agent  *(slides 4–5)*

Every agent follows the same loop:

> **User input → Agent processes → Model generates → Response returned**

The three building blocks in code are the **model**, the **instructions** (the system
prompt that defines its role), and the **response**. Let's meet **Safari**, our travel
assistant.


In [ ]:
from agno.agent import Agent

safari = Agent(
    name="Safari",
    model=make_model(),
    instructions=[
        "You are Safari, a friendly and concise travel-planning assistant.",
        "You specialise in trips across Africa.",
        "Give practical, specific suggestions — not generic tourist fluff.",
    ],
    markdown=True,  # render the answer as nice markdown
)

# print_response runs the agent and pretty-prints the result.
safari.print_response("Suggest 3 things to do in Cape Town for a first-time visitor.")


### Inspecting the response object

`print_response()` is great for demos, but in real apps you call **`.run()`**, which
returns a `RunOutput` object you can inspect programmatically — the text is in
`.content`, plus you get messages, tool calls and token metrics.


In [ ]:
response = safari.run("In one sentence, what's the best time of year to visit Nairobi?")

print("TYPE   :", type(response).__name__)
print("CONTENT:", response.content)
print("TOKENS :", response.metrics)  # input/output token usage


### Streaming

For a responsive UX, stream tokens as they're generated with `stream=True`.


In [ ]:
safari.print_response(
    "Give me a 2-line pitch for why someone should visit Zanzibar.",
    stream=True,
)


---
## 3. Tools: turning an assistant into an *agent*  *(slides 6–7)*

A chatbot only *talks*. An **agent acts** — it calls APIs, queries databases, runs code.
**Tools** are what make that possible.

The loop becomes:

> **Agent decides → Tool executes → External system responds → Result returned to the model**

In Agno, **any Python function is a tool**. The function's name, type hints and docstring
are automatically turned into a schema the model can call. Three flavours:

- **Custom** — your own Python functions (optionally decorated with `@tool`)
- **Built-in** — ready-made toolkits (web search, calculator, file I/O, finance…)
- **External APIs** — wrap any REST/GraphQL/DB call in a function

### 3a. Custom tools

The model can't actually know the live weather — so we give it a tool. (We return fake
data here to keep the workshop offline & free; in real life you'd call a weather API.)


In [ ]:
from agno.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: The name of the city to look up.
    """
    # 🔧 Pretend API call. Swap for a real one (e.g. open-meteo) in production.
    fake_db = {
        "cape town": "18°C, windy and clear",
        "nairobi": "24°C, sunny",
        "marrakech": "31°C, hot and dry",
    }
    return fake_db.get(city.lower(), f"No data for {city}, assume mild and pleasant.")


@tool
def packing_estimate(days: int, climate: str) -> str:
    """Estimate how many outfits to pack.

    Args:
        days: Number of days for the trip.
        climate: A short climate description (e.g. 'hot', 'cold', 'rainy').
    """
    outfits = max(3, round(days * 0.7))
    extra = " Bring layers." if "cold" in climate.lower() else " Pack light, breathable clothes."
    return f"~{outfits} outfits for {days} days in {climate} weather.{extra}"


Now we give these tools to a new agent. Set `debug_mode=True` to *see* the tool calls
happen — invaluable for understanding what the agent is doing under the hood.


In [ ]:
planner = Agent(
    name="Safari Planner",
    model=make_model(),
    tools=[get_weather, packing_estimate],
    instructions=[
        "You are a travel planner. Use your tools to ground answers in real data.",
        "When asked about weather or packing, call the relevant tool instead of guessing.",
    ],
    markdown=True,
)

planner.print_response(
    "I'm going to Marrakech for 5 days. What's the weather like and what should I pack?"
)


Notice the agent **chained two tools**: it called `get_weather("Marrakech")`, used the
result to describe the climate, then called `packing_estimate(5, "hot")` — all on its own.
That decision-making is the essence of an agent.

### 3b. Built-in tools

Agno ships 100+ toolkits. Let's add **live web search** via DuckDuckGo (no API key
needed) so Safari can pull in real, current information.


In [ ]:
from agno.tools.duckduckgo import DuckDuckGoTools

researcher = Agent(
    name="Safari Researcher",
    model=make_model(),
    tools=[DuckDuckGoTools()],
    instructions=[
        "You research travel questions using web search.",
        "Always cite the source (site name) for facts you find.",
        "Be concise.",
    ],
    markdown=True,
)

researcher.print_response(
    "What are 2 well-known annual festivals in Marrakech, and roughly when do they happen?"
)


---
## 4. Structured Outputs: reliable, machine-readable answers  *(slide 8)*

Free-form text is unpredictable. If your agent feeds a database, a UI, or another service,
you need **guaranteed structure**. Agno does this with **Pydantic** models via the
`output_schema` parameter — the model is constrained to return data matching your schema,
and you get back a **validated Python object** (no parsing, no regex).


In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Activity(BaseModel):
    name: str = Field(..., description="Name of the activity or attraction")
    why: str = Field(..., description="One short reason it's worth doing")
    est_cost_usd: int = Field(..., description="Rough cost per person in USD, 0 if free")

class DestinationGuide(BaseModel):
    city: str
    country: str
    best_months: List[str] = Field(..., description="Best months to visit")
    activities: List[Activity] = Field(..., description="3 recommended activities")


In [ ]:
guide_agent = Agent(
    name="Guide Builder",
    model=make_model(),
    instructions="You create concise, structured travel guides.",
    output_schema=DestinationGuide,   # 👈 constrain the output to our schema
)

result = guide_agent.run("Create a short guide for Cape Town, South Africa.")

guide = result.content          # this is a DestinationGuide instance — fully typed!
print(type(guide).__name__)
print(f"\n📍 {guide.city}, {guide.country}")
print(f"🗓️  Best months: {', '.join(guide.best_months)}")
for a in guide.activities:
    print(f"  • {a.name} (~${a.est_cost_usd}) — {a.why}")


Because `guide` is a real Pydantic object, you can use it directly in code — serialise
to JSON, store it, validate it, or render it however you like:


In [ ]:
import json
print(json.dumps(guide.model_dump(), indent=2))


---
## 5. Teams: specialised agents working together  *(slides 9–10)*

Complex tasks benefit from **specialisation**. Instead of one do-everything agent, a
**Team** has a leader that delegates subtasks to focused members and synthesises their
work — often higher quality than any single generalist.

> **Why teams win:** Specialisation (each agent has one job) · Delegation (the leader
> splits the task automatically) · Quality (focused agents beat generalists).

Let's build a 2-member team that produces a mini travel guide:
- a **Researcher** that gathers facts (with web search), and
- a **Writer** that turns those facts into an engaging summary.

Each member gets a `role`; the **team leader** (its own model) coordinates the handoffs.


In [ ]:
from agno.team import Team

research_member = Agent(
    name="Researcher",
    role="Find and verify concrete facts about a destination using web search.",
    model=make_model(),
    tools=[DuckDuckGoTools()],
    instructions="Gather accurate, specific facts. Note the source for each.",
)

writer_member = Agent(
    name="Writer",
    role="Turn research notes into a vivid, well-structured traveller's summary.",
    model=make_model(),
    instructions="Write engaging, concise prose. Use the researcher's facts faithfully.",
)

travel_team = Team(
    name="Travel Guide Team",
    model=make_model(),                 # the leader / orchestrator
    members=[research_member, writer_member],
    instructions=[
        "Coordinate the members to produce a short, factual travel guide.",
        "First have the researcher gather facts, then the writer compose the guide.",
    ],
    show_members_responses=True,        # see each member's contribution
    markdown=True,
)


In [ ]:
travel_team.print_response(
    "Create a short, lively travel guide for a weekend in Nairobi, Kenya.",
    stream=True,
)


You can watch the **leader delegate** to the Researcher and Writer, then merge their
output into a final guide. That orchestration is automatic — you just describe the roles.

---
## ✅ Recap — Foundations

You've built up a full picture of an Agno agent:

1. **Agent = Model + Instructions** — the core request/response loop (`.print_response`, `.run`, streaming).
2. **Tools** make agents *act* — custom `@tool` functions and built-in toolkits like web search.
3. **Structured outputs** give reliable, typed data via Pydantic `output_schema`.
4. **Teams** let specialised agents collaborate under a coordinating leader.

### ➡️ Next: the Advanced Session
Open **`02_agno_advanced.ipynb`** to make agents *stateful and production-ready*:
**Memory** (remembering users), **Knowledge** (RAG over documents), **Storage**
(persistence across restarts), and **Workflows** (multi-step orchestration).
